# 📊 Análise de Sentimento - Unidade D

## Objetivo
Analisar a percepção dos pacientes a partir das avaliações públicas.

## Metodologia
- NLP (análise de sentimento)
- análise temporal
- análise de frequência de palavras

In [23]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from transformers import pipeline

In [24]:
df_bessa = pd.read_csv("03_Bessa_Google_Reviews.csv")
df_bessa.head()

,unidade,autor,rating,data,comentario
0,Bessa,Guilherme de Castro Lima,5,2026,Melhor clínica que já fui.Atendimento sempre r...
1,Bessa,Thiago Lima,1,2026,"Mesmo sendo preferencial, vários representante..."
2,Bessa,Sydney Sabedot,5,2026,"Bom atendimento e profissionais qualificados, ..."
3,Bessa,Chris,4,2026,Todas as vezes que fui a essa clínica gostei b...
4,Bessa,Katia Kelly,4,2025,"Excelente local, aconchegante e acolhedor. Por..."


In [25]:
df_bessa.info()
df_bessa.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   unidade     134 non-null    object
 1   autor       134 non-null    object
 2   rating      134 non-null    int64 
 3   data        134 non-null    int64 
 4   comentario  134 non-null    object
dtypes: int64(2), object(3)
memory usage: 5.4+ KB


(134, 5)

In [26]:
df_bessa["rating"].value_counts().sort_index()
df_bessa["rating"].value_counts(normalize=True).sort_index() * 100

,proportion
rating,
1,25.373134
2,5.970149
3,2.238806
4,9.701493
5,56.716418


In [27]:
df_bessa["data"].value_counts().sort_index()
df_bessa.groupby("data")["rating"].mean().round(2)

,rating
data,
2018,5.00
2019,4.73
2020,2.88
2021,3.75
2022,3.28
2023,3.50
2024,3.10
2025,3.94
2026,3.75


In [28]:
df_bessa["comentario"] = df_bessa["comentario"].fillna("Avaliação sem texto")

df_sent_bessa = df_bessa[df_bessa["comentario"] != "Avaliação sem texto"].copy()
len(df_sent_bessa)

88

In [29]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: pysentimiento/robertuito-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
def analisar_sentimento(texto):
    resultado = sentiment_pipeline(
        str(texto),
        truncation=True,
        max_length=128
    )[0]
    return resultado["label"]

In [31]:
df_crit_bessa = df_sent_bessa[df_sent_bessa["data"].isin([2023, 2024, 2025, 2026])].copy()
len(df_crit_bessa)

46

In [32]:
df_crit_bessa["sentimento"] = df_crit_bessa["comentario"].apply(analisar_sentimento)

df_crit_bessa["sentimento"].value_counts(normalize=True) * 100

,proportion
sentimento,
POS,47.826087
NEG,39.130435
NEU,13.043478


In [33]:
stopwords = [
    "de","da","do","das","dos","a","o","as","os","e","é","em","para",
    "com","que","não","na","no","uma","um","mais","foi","por","se",
    "ao","à","às","como","já","eu","ele","ela","me","minha","meu",
    "muito","porque","isso","está","estava","são","ser","tem","pra",
    "pelo","pela","até","fui","era"
]

def limpar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    palavras = texto.split()
    palavras = [p for p in palavras if p not in stopwords and len(p) > 2]
    return palavras

In [34]:
df_neg_bessa = df_crit_bessa[df_crit_bessa["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg_bessa["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 17),
 ('unimed', 8),
 ('médico', 6),
 ('pessoas', 4),
 ('quem', 4),
 ('clínica', 4),
 ('mesmo', 3),
 ('preferencial', 3),
 ('frente', 3),
 ('apenas', 3),
 ('paciente', 3),
 ('consulta', 3),
 ('dia', 3),
 ('quando', 3),
 ('dor', 3),
 ('péssima', 3),
 ('esperando', 3),
 ('mal', 3),
 ('parece', 3),
 ('falam', 3)]

In [19]:
df_bessa["comentario"] = df_bessa["comentario"].fillna("Avaliação sem texto")

df_sent = df_bessa[df_bessa["comentario"] != "Avaliação sem texto"].copy()

In [20]:
df_bessa["comentario"] = df_bessa["comentario"].fillna("Avaliação sem texto")

df_sent = df_bessa[df_bessa["comentario"] != "Avaliação sem texto"].copy()

In [21]:
df_critico = df_sent[df_sent["data"].isin([2023, 2024, 2025, 2026])].copy()

len(df_critico)

46

In [22]:
df_pos = df_critico[df_critico["sentimento"] == "POS"].copy()

todas_palavras_pos = []

for comentario in df_pos["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras_pos.extend(palavras)

Counter(todas_palavras_pos).most_common(20)

KeyError: 'sentimento'

In [35]:
df_critico.head()

,unidade,autor,rating,data,comentario
0,Bessa,Guilherme de Castro Lima,5,2026,Melhor clínica que já fui.Atendimento sempre r...
1,Bessa,Thiago Lima,1,2026,"Mesmo sendo preferencial, vários representante..."
2,Bessa,Sydney Sabedot,5,2026,"Bom atendimento e profissionais qualificados, ..."
3,Bessa,Chris,4,2026,Todas as vezes que fui a essa clínica gostei b...
4,Bessa,Katia Kelly,4,2025,"Excelente local, aconchegante e acolhedor. Por..."


In [36]:
df_critico["sentimento"] = df_critico["comentario"].apply(analisar_sentimento)

In [37]:
df_critico.head()

,unidade,autor,rating,data,comentario,sentimento
0,Bessa,Guilherme de Castro Lima,5,2026,Melhor clínica que já fui.Atendimento sempre r...,POS
1,Bessa,Thiago Lima,1,2026,"Mesmo sendo preferencial, vários representante...",NEG
2,Bessa,Sydney Sabedot,5,2026,"Bom atendimento e profissionais qualificados, ...",POS
3,Bessa,Chris,4,2026,Todas as vezes que fui a essa clínica gostei b...,POS
4,Bessa,Katia Kelly,4,2025,"Excelente local, aconchegante e acolhedor. Por...",POS


In [38]:
df_pos = df_critico[df_critico["sentimento"] == "POS"].copy()

todas_palavras_pos = []

for comentario in df_pos["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras_pos.extend(palavras)

Counter(todas_palavras_pos).most_common(20)

[('atendimento', 17),
 ('excelente', 8),
 ('clínica', 5),
 ('recepção', 5),
 ('bem', 5),
 ('atendida', 5),
 ('rápido', 4),
 ('bom', 4),
 ('atendente', 4),
 ('dra', 4),
 ('ótima', 4),
 ('médicos', 4),
 ('médico', 3),
 ('estacionamento', 3),
 ('ortopedia', 3),
 ('atenção', 3),
 ('sou', 3),
 ('melhor', 2),
 ('sempre', 2),
 ('ambiente', 2)]

In [39]:
df_neg = df_critico[df_critico["sentimento"] == "NEG"].copy()

todas_palavras = []

for comentario in df_neg["comentario"]:
    palavras = limpar_texto(comentario)
    todas_palavras.extend(palavras)

Counter(todas_palavras).most_common(20)

[('atendimento', 17),
 ('unimed', 8),
 ('médico', 6),
 ('pessoas', 4),
 ('quem', 4),
 ('clínica', 4),
 ('mesmo', 3),
 ('preferencial', 3),
 ('frente', 3),
 ('apenas', 3),
 ('paciente', 3),
 ('consulta', 3),
 ('dia', 3),
 ('quando', 3),
 ('dor', 3),
 ('péssima', 3),
 ('esperando', 3),
 ('mal', 3),
 ('parece', 3),
 ('falam', 3)]

In [40]:
media_por_ano = df_bessa.groupby("data")["rating"].mean()

anos = media_por_ano.index.values
medias = media_por_ano.values

coef = np.polyfit(anos, medias, 1)

pred_2027 = np.polyval(coef, 2027)
pred_2028 = np.polyval(coef, 2028)

pred_2027, pred_2028

(np.float64(3.172626649637948), np.float64(3.0531822051934796))